# PUTM EV — Google Drive Connection Check

This notebook verifies:
1. Whether the Google Drive connection works
2. Whether the required folder structure exists
3. Whether the data files are in place (name, size, row count)
4. Displays a summary — what is OK and what is missing

**Required structure on Google Drive:**
```
MyDrive/
└── MOTORSPORT/
    └── PUTM_EV_FF_CurrentLoop/
        ├── data/
        │   └── model/          ← upload CSV files here
        ├── vcu_models/         ← Colab saves trained models here
        └── raports/            ← Colab saves reports and plots here
```

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive zamontowany.")

## 1b. Diagnostics — verify Drive contents

In [ ]:
import os

root = '/content/drive/MyDrive/MOTORSPORT/PUTM_EV_FF_CurrentLoop'

print(f"Zawartość {root}:")
if os.path.isdir(root):
    for item in sorted(os.listdir(root)):
        print(f"  {item}")
else:
    print("  FOLDER NIE ISTNIEJE — sprawdź nazwę na Drive")

data_path = os.path.join(root, 'data')
if os.path.isdir(data_path):
    print(f"\nZawartość data/:")
    for item in sorted(os.listdir(data_path)):
        print(f"  {item}")

    model_path = os.path.join(data_path, 'model')
    if os.path.isdir(model_path):
        print(f"\nZawartość data/model/:")
        files = sorted(os.listdir(model_path))
        if files:
            for item in files:
                size_mb = os.path.getsize(os.path.join(model_path, item)) / 1024 / 1024
                print(f"  {item:<35} {size_mb:.2f} MB")
        else:
            print("  (pusty)")
    else:
        print("\n  Folder data/model/ NIE ISTNIEJE")
else:
    print("\n  Folder data/ NIE ISTNIEJE")

## 2. Define expected structure

In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/MOTORSPORT/PUTM_EV_FF_CurrentLoop'

# Foldery które muszą istnieć (identyczne jak lokalnie)
REQUIRED_DIRS = {
    'data/model' : 'Dane treningowe (CSV)',
    'vcu_models' : 'Wytrenowane modele (.pkl, .json)',
    'raports'    : 'Raporty, wykresy, Excel z metrykami',
}

# Pliki CSV które muszą być w data/model/
REQUIRED_FILES = [
    'final_data.csv',
    'fsp_endu_current.csv',
    'setpoints.csv',
    'merged_wheel_data.csv',
    'front_right_data.csv',
    'rear_left_data.csv',
    'rear_right_data.csv',
]

# Oczekiwana minimalna liczba wierszy (bez nagłówka)
MIN_ROWS = {
    'final_data.csv'       : 50_000,
    'fsp_endu_current.csv' : 25_000,
    'setpoints.csv'        : 50_000,
    'merged_wheel_data.csv': 15_000,
    'front_right_data.csv' : 15_000,
    'rear_left_data.csv'   : 12_000,
    'rear_right_data.csv'  : 10_000,
}

print(f"Korzeń projektu: {DRIVE_ROOT}")

## 3. Check and create folders

In [ ]:
dir_status = {}

for rel_path, description in REQUIRED_DIRS.items():
    full_path = os.path.join(DRIVE_ROOT, rel_path)
    existed = os.path.isdir(full_path)
    if not existed:
        os.makedirs(full_path, exist_ok=True)
        status = 'UTWORZONO'
    else:
        status = 'OK'
    dir_status[rel_path] = (status, full_path, description)
    icon = '✓' if existed else '⚠'
    print(f"  {icon}  [{status:8s}]  {rel_path}/  —  {description}")

print()
print("Foldery gotowe.")

## 4. Check data files

In [ ]:
import pandas as pd

DATA_DIR = os.path.join(DRIVE_ROOT, 'data', 'model')

file_status = []

print(f"Sprawdzam pliki w: {DATA_DIR}\n")
print(f"  {'Plik':<28} {'Status':<10} {'Rozmiar':>10}  {'Wiersze':>10}  {'Kolumny'}")
print(f"  {'-'*28} {'-'*10} {'-'*10}  {'-'*10}  {'-'*30}")

for fname in REQUIRED_FILES:
    fpath = os.path.join(DATA_DIR, fname)
    row = {'plik': fname}

    if not os.path.isfile(fpath):
        row.update({'status': 'BRAK', 'rozmiar': '-', 'wiersze': '-', 'kolumny': '-'})
        print(f"  ✗  {fname:<28} {'BRAK':<10} {'':>10}  {'':>10}")
    else:
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        try:
            df = pd.read_csv(fpath, nrows=0)  # tylko nagłówek
            cols = list(df.columns)
            # liczymy wiersze bez wczytywania całości
            with open(fpath, 'r') as f:
                n_rows = sum(1 for _ in f) - 1  # -1 nagłówek
            min_req = MIN_ROWS.get(fname, 0)
            ok = n_rows >= min_req
            status = 'OK' if ok else 'MAŁO DANYCH'
            icon = '✓' if ok else '⚠'
            row.update({'status': status, 'rozmiar': f'{size_mb:.2f} MB',
                        'wiersze': n_rows, 'kolumny': ', '.join(cols)})
            print(f"  {icon}  {fname:<28} {status:<10} {size_mb:>8.2f}MB  {n_rows:>10,}  {', '.join(cols[:6])}{'...' if len(cols)>6 else ''}")
        except Exception as e:
            row.update({'status': 'BŁĄD', 'rozmiar': f'{size_mb:.2f} MB', 'wiersze': '?', 'kolumny': str(e)})
            print(f"  ✗  {fname:<28} {'BŁĄD':<10} {size_mb:>8.2f}MB  —  {e}")

    file_status.append(row)

## 5. Check saved models (if any)

In [ ]:
import datetime

MODELS_DIR = os.path.join(DRIVE_ROOT, 'vcu_models')

model_files = [
    f for f in os.listdir(MODELS_DIR)
    if f.endswith(('.pkl', '.json', '.h5', '.pt', '.onnx'))
] if os.path.isdir(MODELS_DIR) else []

if model_files:
    print(f"Znalezione modele w {MODELS_DIR}:")
    for mf in sorted(model_files):
        mpath = os.path.join(MODELS_DIR, mf)
        size_mb = os.path.getsize(mpath) / 1024 / 1024
        mtime_str = datetime.datetime.fromtimestamp(os.path.getmtime(mpath)).strftime('%Y-%m-%d %H:%M')
        print(f"  ✓  {mf:<40} {size_mb:>8.2f} MB   zapisany: {mtime_str}")
else:
    print("  (brak zapisanych modeli — folder jest pusty, to normalne przed pierwszym treningiem)")

## 6. Summary

In [ ]:
missing_files = [r['plik'] for r in file_status if r['status'] == 'BRAK']
broken_files  = [r['plik'] for r in file_status if r['status'] not in ('OK', 'BRAK')]
ok_files      = [r['plik'] for r in file_status if r['status'] == 'OK']
new_dirs      = [p for p, (s, _, _) in dir_status.items() if s == 'UTWORZONO']

print("=" * 60)
print("  PODSUMOWANIE")
print("=" * 60)

print(f"\n  Foldery:")
print(f"    OK        : {len(REQUIRED_DIRS) - len(new_dirs)}/{len(REQUIRED_DIRS)}")
if new_dirs:
    print(f"    Utworzono : {', '.join(new_dirs)}")

print(f"\n  Pliki danych:")
print(f"    OK        : {len(ok_files)}/{len(REQUIRED_FILES)}")

if missing_files:
    print(f"\n  BRAKUJĄCE PLIKI (musisz wgrać na Drive):")
    for f in missing_files:
        print(f"    ✗  {f}")
    print(f"\n  Skopiuj brakujące pliki do:")
    print(f"    {DATA_DIR}")

if broken_files:
    print(f"\n  PLIKI Z PROBLEMEM:")
    for f in broken_files:
        r = next(x for x in file_status if x['plik'] == f)
        print(f"    ⚠  {f}  —  {r['status']}")

print()
if not missing_files and not broken_files:
    print("  ✓ Wszystko gotowe — możesz przejść do notebooka treningowego.")
else:
    print("  ✗ Uzupełnij brakujące pliki przed uruchomieniem treningu.")

print("=" * 60)
print(f"\n  Ścieżki do użycia w notebooku treningowym:")
print(f"    DATA_DIR    = '{DATA_DIR}'")
print(f"    MODELS_DIR  = '{os.path.join(DRIVE_ROOT, 'vcu_models')}'")
print(f"    REPORTS_DIR = '{os.path.join(DRIVE_ROOT, 'raports')}'")